In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import json
import itertools
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pandas import DataFrame
from sklearn.metrics import confusion_matrix

OUTPUT_DATA_DIR = os.environ.get('OUTPUT_DATA_DIR', '../output_data')
SOURCE_DATA_DIR = os.environ.get('SOURCE_DATA_DIR', '../source_data')

## Feature matrix

In [ ]:
data = np.load(f'{OUTPUT_DATA_DIR}/connectomes.npz', allow_pickle=True)
X, y = data['X'], data['y']
print(f'{len(X)} subjects, {X.shape[1]} features (ROI pairs)')

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(X, aspect='auto', interpolation='nearest')
plt.colorbar(im, ax=ax)
ax.set_title('Functional connectivity feature matrix')
ax.set_xlabel('Features (ROI pairs)')
ax.set_ylabel('Subjects')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DATA_DIR}/feature_matrix.png', dpi=150)
plt.show()

## Classification results

In [ ]:
with open(f'{OUTPUT_DATA_DIR}/classification_results.json') as f:
    results = json.load(f)

print('Cross-validation accuracy per fold:', results['cv_accuracy_per_fold'])
print(f"CV mean accuracy:   {results['cv_accuracy_mean']:.3f}")
print(f"Test accuracy:      {results['test_accuracy']:.3f}")
print(f"Permutation p-value: {results['permutation_pvalue']:.4f}")

In [ ]:
preds = np.load(f'{OUTPUT_DATA_DIR}/classification_predictions.npz', allow_pickle=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

for ax, y_true, y_pred, title in [
    (ax1, preds['y_train'], preds['y_pred_cv'], 'Training set (3-fold CV)'),
    (ax2, preds['y_test'],  preds['y_pred_test'], 'Test set (held-out)'),
]:
    cm = confusion_matrix(y_true, y_pred)
    cmdf = DataFrame(cm, index=['Adult', 'Child'], columns=['Adult', 'Child'])
    sns.heatmap(cmdf, cmap='RdBu_r', ax=ax, annot=False)
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        ax.text(j + 0.5, i + 0.5, str(cm[i, j]), ha='center', color='white', fontsize=14)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Observed')
    ax.set_title(title)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DATA_DIR}/confusion_matrices.png', dpi=150)
plt.show()

## Feature importance (SVC weights on brain)

In [ ]:
from nilearn import datasets, plotting

feat_matrix = preds['feat_matrix']
basc = datasets.fetch_atlas_basc_multiscale_2015(data_dir=SOURCE_DATA_DIR, resolution=64)
coords = plotting.find_parcellation_cut_coords(basc.maps)

display = plotting.plot_connectome(
    feat_matrix, coords, colorbar=True, edge_threshold=0.005,
    title='SVC feature importance (BASC 64 parcellation)'
)
display.savefig(f'{OUTPUT_DATA_DIR}/connectome_weights.png')
display.close()

plotting.plot_matrix(
    feat_matrix, figure=(8, 7),
    labels=range(feat_matrix.shape[0]),
    reorder=False, tri='lower'
)
plt.savefig(f'{OUTPUT_DATA_DIR}/feature_importance_matrix.png', dpi=150)
plt.show()